# Homework10

Exercises with text processing and NLP modeling

## Goals

- Understand similarities and differences between the processes of working with text, images and tabular data
- Practice with different methods of encoding and modeling text data
- See different methods for extracting information or patterns from text datasets

### Setup

Run the following 2 cells to import all necessary libraries and helpers for this homework.

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/text_utils.py

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from data_utils import display_silhouette_plots, object_from_json_url
from text_utils import get_top_words

You can tell it's gonna be a good homework from the number of imports.
# 🙃

## Cheat Sheet

There are lots of conversions in this homework between lists, strings, vectors, scores, etc...

This list can help as a quick reference for the names of the functions and what they expect as input, and produce as output.

These all operate on lists of _something_ and return lists of _something else_. But, here, let's just focus on the _type_ of the elements that go in these lists. Most of them will work on single elements, but not all.

`TFIDF.fit_transform(string)` / `TFIDF.transform(string)` : string -> sparse vector

`TFIDF.inverse_transform(vector)` : sparse vector -> list of words (NOT a string)

`TFIDF.get_feature_names_out()` : None -> list of words (vocab of dataset)

`TFIDF.get_feature_names_out()[idx]` : int (column index / word id) -> word

`vector.toarray()` : vector -> regular array

`get_top_words(vector, vocab, n_words=N)` : vector + list of all words + int -> list of top-N words in vector

`argsort(metric)` : list of floats -> list of int indexes

For example:

To get the words in a sentence ordered by importance (most important first), we have to turn a string into a vector, then the vector into a regular list of scores, then the list into indexes, and, finally, fetch words from our vocab list using these indexes.

```py
sentence = "a sentence goes here"
num_words = len(sentence.split(" "))
sentence_vct = TFIDF.transform([sentence])
sentence_vct_list = sentence_vct.toarray()
idx_by_importance = argsort(sentence_vct_list, reverse=True)[:num_words]
TFIDF.get_feature_names_out()[idx_by_importance]
```

For this particular example we could've used the `get_top_words()` function to get words ordered by importance:

```py
sentence = "a sentence goes here"
num_words = len(sentence.split(" "))

get_top_words(
  TFIDF.transform([sentence]),
  TFIDF.get_feature_names_out(),
  n_words=num_words
)
```

## Have protein, need seasoning

Let's create a model to help us season our foods. In the end, what we want is a model that receives a short list of ingredients and returns a list of seasonings or complementary ingredients for our original ingredients list.

In order to do that we need a dataset of recipes. We'll load that into a text dataset where each recipe is a document and the ingredients are our document *tokens*.

Let's take a look at the recipe dataset and become familiar with the data and how it's organized.

We'll load our recipes and do a bit of exploratory data analysis to look for patterns first to see if this kind of modeling makes any sense.

### Load Data

Here's our dataset. Let's load it into an object for inspection:

In [ ]:
DATA_PATH = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/refs/heads/main/datasets/text/recipes/recipes_min16.json"
recipes_obj = object_from_json_url(DATA_PATH)

### Look at Data

How's the data organized?

How many recipes do we have?

Do all recipes have the same number of ingredients?

Anything else stand out about the data?

In [ ]:
# TODO: Look at Data here
print(f"First recipe: {recipes_obj[0]}")
print(f"Second recipe: {recipes_obj[1]}")
# TODO: How many recipes
print(f"Number of recipes: {len(recipes_obj)}")

# TODO: How many ingredients do the shortest and longest recipes have?
ingredient_counts = [len(recipe["ingredients"]) for recipe in recipes_obj]
print(f"Shortest recipe has {min(ingredient_counts)} ingredients.")
print(f"Longest recipe has {max(ingredient_counts)} ingredients.")
print(f"Average number of ingredients: {sum(ingredient_counts) / len(ingredient_counts):.2f}")

### Create Input Features

Our dataset doesn't really have to be a `DataFrame` here. It can, but it doesn't have to be.

Each recipe right now is described as a list of ingredients, but what we really want is a list of *sentences*, where each *sentence* is a Python `string` with all of the ingredients for a given recipe.

Instead of:<br>```["salt", "baking soda", "water", "mushroom"]```,

we want:<br>```"salt baking soda water mushroom"```

The `join()` function might help.

Another thing to consider is wether we want to do anything special about multi-word ingredients, like *baking soda*.

Do we want to let our vectorizer split that into two tokens, or do we want to guarantee that *baking* and *soda* always stay together? 

In [ ]:
# TODO: turn list of objects into list of strings
def ingredients_to_string(ingredient_list):
    return " ".join([ing.replace(" ", "_") for ing in ingredient_list])

recipes_strings = [ingredients_to_string(r["ingredients"]) for r in recipes_obj]

print(f"total characters in all recipes: {sum(len(s) for s in recipes_strings)}")
print("example:", recipes_strings[0])

### Encode Data

The fun part.

Let's vectorize our list of ingredient strings into a sparse document matrix using `CountVectorizer` or `TfidfVectorizer`.

The resulting matrix will have one row for each recipe, and the columns will encode the ingredients.

In [ ]:
# TODO: Vectorize ingredients from our recipe list
TFIDF = TfidfVectorizer(min_df=1, strip_accents='unicode', lowercase=True)
recipes_vct = TFIDF.fit_transform(recipes_strings)

vocab = TFIDF.get_feature_names_out()

# TODO: How many words are in our vocabulary?
print(f"Vocabulary size: {len(vocab)}")
print(f"Vector matrix shape: {recipes_vct.shape}")
print("Vocabulary sample:", vocab[:20])

### Cluster Data

Now that we have our recipes/documents vectorized we can study them a little bit, and look for patterns.

What happens if we cluster our recipes ? What do the cluster centers represent ?

When might this be useful ?

In [ ]:
# TODO: cluster recipes
import numpy as np
mClust = KMeans(n_clusters=8, random_state=800)
recipe_clusters = mClust.fit_predict(recipes_vct)

print(f"Number of clusters: {mClust.n_clusters}")
print(f"cluster_centers_ shape: {mClust.cluster_centers_.shape}")

# Check number of recipes per cluster
unique, counts = np.unique(recipe_clusters, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  Cluster {c}: {n} recipes")

### Cluster Centers

Use the `get_top_words()` function to decode the `cluster_centers` back into ingredients.

In [ ]:
# TODO: Look at cluster centers
top_words_per_cluster = get_top_words(mClust.cluster_centers_, vocab, n_words=8)
print("Cluster center ingredients:")
for i, words in enumerate(top_words_per_cluster):
    print(f"  Cluster {i}: {words}")

### Interpretation

<span style="color:hotpink">
What do these cluster centers represent ?<br>
Is there anything interesting about recipe cluster centers ?<br>
</span>

<span style="color:hotpink;">The clusters probably show different cooking styles. Like baking stuff in one cluster, Asian food in another. Common ingredients like salt show up in many clusters.</span>

### Plot Clusters

Let's plot our clusters to see if we have to adjust any of the clustering parameters.

Since we can't plot in $500$ dimensions, we should use `TruncatedSVD` to look at our clusters in $2D$ and $3D$.

In [ ]:
# TODO: TruncatedSVD to reduce the dimensions of our feature space
svd = TruncatedSVD(n_components=3, random_state=1010)
recipes_svd = svd.fit_transform(recipes_vct)

print(f"SVD explained variance: {sum(svd.explained_variance_ratio_):.3f}")

# plot clusters - 2D
plt.figure(figsize=(10, 6))
plt.scatter(recipes_svd[:, 0], recipes_svd[:, 1],
            c=recipe_clusters, marker='o', alpha=0.5, s=5, cmap='tab10')
plt.xlabel("SVD Component 0")
plt.ylabel("SVD Component 1")
plt.title("Recipe Clusters (2D via TruncatedSVD)")
plt.colorbar(label="Cluster")
plt.show()

# TODO: plot clusters
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(recipes_svd[:, 0], recipes_svd[:, 1], recipes_svd[:, 2],
           c=recipe_clusters, marker='o', alpha=0.5, s=5, cmap='tab10')
ax.set_xlabel("SVD 0")
ax.set_ylabel("SVD 1")
ax.set_zlabel("SVD 2")
ax.set_title("Recipe Clusters (3D via TruncatedSVD)")
plt.show()

### Interpretation

<span style="color:hotpink">
What does the graph look like ?<br>
Are the clusters well-separated ?
</span>

<span style="color:hotpink">Clusters are bunched in the middle. Hard to separate because recipes share common ingredients like salt.</span>

### Plot Silhouette Plots

We can also check the quality of our clustering by looking at the silhouette plots that we get from calling:<br>
`display_silhouette_plots(vectors, clusters)`.

In [ ]:
display_silhouette_plots(recipes_vct, recipe_clusters)

### Interpretation

<span style="color:hotpink">
How many clusters did you end up with ?<br>
How do they look ?<br>
</span>

<span style="color:hotpink">8 clusters. Low silhouette scores, but that's normal for text data.</span>

## Recipe Completion

Ok. On to the main event.

Let's create some recipes.

We'll do this using a technique similar to what is used for movie/product recommendations. Given an initial set of ingredients, we'll look at recipes that have similar ingredients and "recommend" additional ingredients.

We already have all of the recipes in our dataset encoded as `tf-idf` vectors. The rest of our algorithm will be something like:
1. Start with an initial set of ingredients
2. Encode ingredients
3. Find a set of recipes that are similar to our list of ingredients
4. Find common ingredients that are in the similar recipes, but not in our list of ingredients
5. Pick representative ingredient to add to recipe
6. Repeat

Let's start.

### 1. Initial list of ingredients

This is just a string with ingredients:

In [ ]:
recipe_seed_str = "garlic" # feel free to change this

### 2. Encode ingredients

Transform the string into a `tf-idf` vector using the `transofrm()` function from the pre-fitted `TfidfVectorizer` object:

In [ ]:
# TODO: transform string into sparse vector
recipe_seed_vct = TFIDF.transform([recipe_seed_str])

print(f"Recipe seed vector shape: {recipe_seed_vct.shape}")

### 3. Find similar recipes

The meat of the algorithm. No pun intended.

In order to find similar recipes, we'll first calculate the distance between our current list of ingredients and all recipes in our dataset.

We can start with euclidean distance and later try other kinds, but the overall processing will be the same:

1. Start with an empty list to store distances
2. Loop over the `tf-idf` recipe vectors and for each vector:
   1. Subtract the ingredient list
   2. Square the difference (to square a sparse matrix `A`, use `A.multiply(A)`)
   3. Sum the terms of the result
   4. Take the square root of the sum
   5. Append to distance list
3. Find the indices of the smallest distances (this operation is called `argsort` and will give us the indices of the recipes that are most similar to our list of ingredients)
4. Check the recipes to see if they are indeed similar (`inverse_transform()` the vectors at the indices calculated above)

In [ ]:
# argsort a list (get sequence of indices that would sort the list)
# https://stackoverflow.com/a/3382369
def argsort(L, reverse=False):
  L = L[0] if len(L) == 1 else L
  return sorted(range(len(L)), key=L.__getitem__, reverse=reverse)

In [ ]:
# TODO: list to keep distances
recipe_distances = []

# TODO: loop over vectors and append euclidean distances to list
for recipe_vct in recipes_vct:
    diff = recipe_vct - recipe_seed_vct
    sq_diff = diff.multiply(diff)
    sum_sq = sq_diff.sum()
    distance = np.sqrt(sum_sq)
    recipe_distances.append(distance)

# TODO: argsort list of distances to find indices of similar recipes
similar_indices = argsort(recipe_distances)

# TODO: check first 4 recipes
print("Top 4 most similar recipes to garlic:")
for i in range(4):
    idx = similar_indices[i]
    print(f"{i+1}. {recipes_strings[idx]}")

### 4. Find ingredients to recommend

We have a way to get a set of similar recipes with similar ingredients, and now want to find a *meaningful*, or *representative*, ingredient to add to our ingredients list.

Let's consider ingredients in the $16$ most similar recipes. What we are trying to do is find an ingredient that is in a lot of these recipes, but not yet in our list of ingredients.

There are many possible ways of doing this. We could count the number of times different ingredients show up in these $16$ recipes using Python dictionaries and/or sets, but what we're trying to do here is very similar to what a `TfidfVectorizer` does: calculate relative importance of terms in a series of documents.

Let's re-encode these $16$ recipes using their own separate `TfidfVectorizer`, then sum the importance of each ingredient and look at ingredients with the highest importance scores.

We could re-use the vectors/scores from the original `TfidfVectorizer`, but they're gonna be influenced by the relative frequencies of all of the ingredients that showed up in all of the recipes. Using a separate vectorizer is a little bit more precise.

The steps we need to take are:

1. Separate the $16$ recipes most similar to our list of ingredients
   1. We have lots of representations of our recipes, but `recipes` (list of strings) might be the easiest one to use here
2. Create a new `TfidfVectorizer` and encode the $16$ recipes
3. Sum the resulting vectors by column to get overall importance scores for each ingredient/token
4. Convert resulting vector to a list using `A.tolist()[0]`
5. `argsort` the importance scores to get sequence of ingredient indices ordered from most to least important
6. Find the most important ingredient that isn't on the ingredient list

In [ ]:
# TODO: Get 16 most similar recipes
top16_idxs = similar_indices[:16]
top16_recipes = [recipes_strings[idx] for idx in top16_idxs]

# TODO: Encode the 16 recipes
local_tfidf = TfidfVectorizer()
top16_vectors = local_tfidf.fit_transform(top16_recipes)
local_vocab = local_tfidf.get_feature_names_out()

# TODO: Sum the recipe vectors by column to get ingredient importance scores
importance_scores = top16_vectors.sum(axis=0)

# TODO: Convert sparse vector to regular list with A.tolist()[0]
importance_list = importance_scores.A1

# TODO: argsort the importance scores
ingredient_indices = argsort(importance_list, reverse=True)

# TODO: Find most important ingredient not yet on the list of ingredients
current_ingredients = set(recipe_seed_str.split())
for idx in ingredient_indices:
    ingredient = local_vocab[idx]
    if ingredient not in current_ingredients:
        print(f"Most important ingredient to add: {ingredient}")
        recommended_ingredient = ingredient
        break

print(f"Current ingredients: {current_ingredients}")
print(f"Recommended ingredient to add: {recommended_ingredient}")

### 5. Add ingredient to recipe

This is simply adding a word to `recipe_seed_str`

In [ ]:
# TODO: add the first important ingredient to list of ingredients
recipe_seed_str = recipe_seed_str + " " + recommended_ingredient
print(f"Updated recipe seed string: {recipe_seed_str}")

### 6. Repeat (Optional)

Now we can repeat this process until we get an empty list of important ingredients: 
1. Encode current recipe
2. Find similar recipes
3. Find important ingredients
4. Add important ingredient

Might be helpful to define a couple of functions, like `find_similar_recipes()` and `find_important_ingredients()`...

Only do this step if you're really curious about experimenting with generating unconventional ingredient lists. It's not going to be graded.

In [ ]:
# TODO: Create find_similar_recipes(ingredients, recipes, vectorizer)

# TODO: Create find_important_ingredients(recipes)

# TODO: Create recipe by repeating calls to find_similar_recipes() and find_important_ingredients()